# Taller 6 — Arquitectura Lakehouse
## Pipeline Bronze → Silver → Gold con Spark, Iceberg y Trino

### Objetivos de aprendizaje

Al terminar este taller el estudiante sera capaz de:

1. **Comprender** la arquitectura Medallion (Bronze/Silver/Gold) y por que cada capa tiene un proposito distinto.
2. **Ejecutar** un pipeline de datos completo: ingestión cruda → limpieza y enriquecimiento → agregacion de metricas.
3. **Ingestar lotes sucesivos** de datos usando `append` en tablas Iceberg y observar el versionado por snapshots.
4. **Consultar** las tablas Iceberg con Trino sin necesidad de Spark, entendiendo el rol del catalogo REST compartido.

### Arquitectura del sistema

```
  [CSV raw]                        (disco local / bucket raw)
      │
      ▼
  Spark (ingest_bronze.py)
      │  agrega ingestion_ts, source_file
      ▼
  Bronze  ─── Iceberg / MinIO  (s3://warehouse/lakehouse/bronze_ventas/)
      │
      ▼
  Spark (transform_silver.py)
      │  limpieza + join dimensiones + total + ganancia
      ▼
  Silver  ─── Iceberg / MinIO  (s3://warehouse/lakehouse/silver_ventas/)
      │
      ▼
  Spark (transform_gold.py)
      │  agrupacion por region y mes
      ▼
  Gold    ─── Iceberg / MinIO  (s3://warehouse/lakehouse/gold_metricas_region/)
      │
      ├──► Trino CLI / Trino UI  (analistas SQL)
      └──► Spark SQL             (ingenieros de datos)

  REST Catalog (iceberg-rest:8181) ← compartido entre Spark y Trino
```


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Taller6-Lakehouse")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.type", "rest")
    .config("spark.sql.catalog.demo.uri", "http://rest:8181")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000")
    .config("spark.sql.defaultCatalog", "demo")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("SparkSession lista.")

## Arquitectura Lakehouse: Bronze, Silver, Gold

La arquitectura **Medallion** (o arquitectura de capas) organiza los datos en zonas con niveles crecientes de calidad:

- **Bronze (Capa de aterrizaje)**
  - Datos crudos tal como llegan de la fuente (CSV, JSON, mensajes de Kafka, etc.).
  - Se agregan columnas de auditoria (`ingestion_ts`, `source_file`) pero NO se modifica el contenido.
  - Permite rereprocesar desde cero si se detecta un error aguas abajo.
  - Soporta multiples appends; cada lote genera un nuevo snapshot en Iceberg.

- **Silver (Capa de datos limpios)**
  - Se eliminan registros invalidos (nulls, valores negativos).
  - Se hace join con tablas de dimensiones (p. ej., proveedor y margen de cada producto).
  - Se calculan metricas de transaccion (`total`, `ganancia`).
  - Nivel de granularidad: una fila = una venta individual.

- **Gold (Capa de metricas de negocio)**
  - Datos pre-agregados, optimizados para consultas analiticas rapidas.
  - Una fila = metricas de una region + categoria + mes.
  - Consumida directamente por dashboards, herramientas de BI o Trino.


In [ ]:
# Explorar el catalogo
print("Namespaces disponibles:")
spark.sql("SHOW NAMESPACES IN demo").show()

print("Tablas en demo.lakehouse:")
spark.sql("SHOW TABLES IN demo.lakehouse").show()

## Capa Bronze: datos crudos

La capa Bronze contiene los datos tal como fueron ingestados, mas las columnas de auditoria.
No hay transformacion de negocio — esto garantiza que siempre se puede rereprocesar.


In [ ]:
bronze = spark.sql("SELECT * FROM demo.lakehouse.bronze_ventas")

print("Schema de Bronze:")
bronze.printSchema()

print("Primeras 5 filas:")
bronze.show(5)

print(f"Total filas Bronze: {bronze.count()}")

## Capa Silver: datos limpios y enriquecidos

Silver agrega valor sobre Bronze de tres formas:
1. **Limpieza**: elimina filas con nulls y valores invalidos.
2. **Enriquecimiento**: join con dimensiones de productos (proveedor, margen).
3. **Calculo**: columnas derivadas `total` y `ganancia`.


In [ ]:
silver = spark.sql("SELECT * FROM demo.lakehouse.silver_ventas")

print("Schema de Silver:")
silver.printSchema()

print("Primeras 5 filas (columnas clave):")
silver.select(
    "id", "fecha", "producto", "proveedor", "categoria",
    "cantidad", "precio_unitario", "total", "margen_pct", "ganancia", "region"
).show(5)

print(f"Total filas Silver: {silver.count()}")

# Columnas nuevas vs Bronze
bronze_cols = set(spark.sql("SELECT * FROM demo.lakehouse.bronze_ventas LIMIT 1").columns)
silver_cols = set(silver.columns)
nuevas = silver_cols - bronze_cols
print(f"\nColumnas nuevas en Silver (no existen en Bronze): {sorted(nuevas)}")

## Capa Gold: metricas de negocio

Gold consolida los datos de Silver en metricas agregadas por **region, categoria y mes**.
Es la capa que consumen los dashboards de negocio y las consultas de Trino.


In [ ]:
gold = spark.sql("SELECT * FROM demo.lakehouse.gold_metricas_region")

print("Schema de Gold:")
gold.printSchema()

print("Metricas por region (consolidado):")
spark.sql("""
    SELECT region,
           SUM(ingresos_totales)  AS ingresos_totales,
           SUM(ganancia_total)    AS ganancia_total,
           SUM(num_transacciones) AS num_transacciones,
           SUM(unidades_vendidas) AS unidades_vendidas
    FROM demo.lakehouse.gold_metricas_region
    GROUP BY region
    ORDER BY ingresos_totales DESC
""").show()

print("Metricas por categoria:")
spark.sql("""
    SELECT categoria,
           SUM(ingresos_totales) AS ingresos_totales,
           SUM(ganancia_total)   AS ganancia_total
    FROM demo.lakehouse.gold_metricas_region
    GROUP BY categoria
    ORDER BY ingresos_totales DESC
""").show()

## Ingestar un segundo lote (append)

En produccion los datos llegan en lotes sucesivos.
Iceberg permite hacer `append` a una tabla existente, generando un nuevo **snapshot**
sin sobrescribir los datos anteriores.

Esto es diferente a `createOrReplace`, que sobrescribe la tabla completa.


In [ ]:
# Simular llegada de lote 2 (datos del Q2 2024)
df_lote2 = spark.read.csv("/home/iceberg/data/ventas_lote2.csv", header=True, inferSchema=True)
df_lote2 = (
    df_lote2
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.lit("ventas_lote2.csv"))
)

df_lote2.writeTo("demo.lakehouse.bronze_ventas").append()

total_bronze = spark.sql("SELECT COUNT(*) FROM demo.lakehouse.bronze_ventas").collect()[0][0]
print(f"Lote 2 ingestado exitosamente.")
print(f"Total filas en Bronze ahora: {total_bronze}")
print("  (lote 1: 30 filas, lote 2: 20 filas -> total esperado: 50)")

In [ ]:
# Ver historial de snapshots de Bronze
# Cada append o createOrReplace genera un snapshot nuevo
print("Historial de snapshots de bronze_ventas:")
spark.sql("""
    SELECT snapshot_id,
           committed_at,
           operation,
           summary['added-records']    AS registros_agregados,
           summary['total-records']    AS total_registros
    FROM demo.lakehouse.bronze_ventas.snapshots
    ORDER BY committed_at
""").show(truncate=False)

print("\nDistribucion por source_file:")
spark.sql("""
    SELECT source_file, COUNT(*) AS filas
    FROM demo.lakehouse.bronze_ventas
    GROUP BY source_file
""").show()

## Consultas con Trino

**Trino** es un motor de consultas SQL distribuido que puede leer directamente las tablas Iceberg
a traves del catalogo REST. No necesita Spark.

### Como conectar a Trino CLI

Desde tu terminal (fuera del notebook):

```bash
# Opcion 1: usando make
make query

# Opcion 2: entrando al contenedor
docker compose exec trino trino --catalog iceberg --schema lakehouse
```

### Queries de ejemplo en Trino CLI

```sql
-- Ver tablas disponibles
SHOW TABLES;

-- Metricas por region
SELECT region, ingresos_totales, num_transacciones
FROM gold_metricas_region
ORDER BY ingresos_totales DESC;

-- Ingresos por categoria (Trino tambien puede agregar sobre Gold)
SELECT categoria, SUM(ingresos_totales) AS ingresos_totales
FROM gold_metricas_region
GROUP BY categoria
ORDER BY ingresos_totales DESC;

-- Consultar Silver directamente
SELECT producto, SUM(total) AS ventas, SUM(ganancia) AS ganancias
FROM silver_ventas
GROUP BY producto
ORDER BY ventas DESC;
```

### Trino UI
Disponible en **http://localhost:8085** — puedes ver el query plan y los stages de ejecucion.


## Preguntas de analisis y cierre

### Preguntas sobre el pipeline

1. ¿Cuantos archivos Parquet hay en cada capa dentro de MinIO? ¿Por que difieren en numero?
2. ¿Que pasa si se ejecuta `ingest_bronze.py` dos veces seguidas? ¿Cuantos registros queda?
3. ¿Que diferencia hay entre `writeTo(...).createOrReplace()` y `writeTo(...).append()`?
4. ¿Por que Silver tiene menos filas que Bronze en un caso de limpieza real?
5. ¿Que informacion pierde Gold respecto a Silver? ¿Cuando ese costo es aceptable?

### Preguntas sobre Trino

6. ¿Por que Trino puede leer tablas Iceberg sin Spark?
7. ¿Que rol juega el catalogo REST (`iceberg-rest`) entre Spark y Trino?
8. ¿En que caso usarias Trino en lugar de Spark SQL?

### Cierre del curso — progresion de los talleres

| Taller | Storage      | Engine          | Formato   | Concepto clave                         |
|--------|--------------|-----------------|-----------|----------------------------------------|
| T2     | HDFS         | MapReduce       | Text/CSV  | Procesamiento distribuido basico       |
| T3     | HDFS         | Spark           | Text/CSV  | Spark como reemplazo de MapReduce      |
| T4     | HDFS / local | Spark           | Parquet   | Formatos columnar y DataFrames         |
| T5     | MinIO (S3)   | Spark           | Iceberg   | Open table formats, snapshots, schema  |
| T6     | MinIO (S3)   | Spark + Trino   | Iceberg   | Lakehouse: capas, catalogo compartido  |

**Reflexion final**: ¿Que habria que cambiar si el storage fuera GCS o Azure Blob en lugar de MinIO?
